In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings_1024=OpenAIEmbeddings(model="text-embedding-3-large",dimensions=1024)
embeddings_1024

c:\Users\OmkarIngale\Desktop\Training\RAG project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001F3CB360EC0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001F3CB361E80>, model='text-embedding-3-large', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [4]:
from pathlib import Path

from langchain_community.document_loaders import (
    Docx2txtLoader,
    PyPDFLoader,
)

def load_documents(folder_path):
    documents = []

    for file_path in Path(folder_path).rglob("*"):
        if not file_path.is_file():
            continue

        try:
            if file_path.suffix.lower() == ".docx":
                loader = Docx2txtLoader(str(file_path))

            elif file_path.suffix.lower() == ".pdf":
                loader = PyPDFLoader(str(file_path))

            else:
                continue

            docs = loader.load()

            # Add source metadata
            for doc in docs:
                doc.metadata["source"] = str(file_path)
                doc.metadata["filename"] = file_path.name

            documents.extend(docs)

        except Exception as e:
            print(f"Error loading {file_path}: {e}")

    print(f"Total documents loaded: {len(documents)}")
    return documents

C:\Users\OmkarIngale\AppData\Local\Temp\ipykernel_50200\597905589.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [5]:
documents = load_documents('documents')

Error loading documents\~$RECTV Holidays and Holiday Pay Policy Management U.S._July 2023 FINAL.docx: File is not a zip file
Total documents loaded: 28


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
final_documents=text_splitter.split_documents(documents)
final_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2023-01-24T18:23:58-08:00', 'author': 'KD984D', 'moddate': '2023-01-24T18:23:58-08:00', 'title': 'Microsoft Word - 2022 Success Bonus FAQ - US', 'source': 'documents\\2022 Success Bonus FAQ (US).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': '2022 Success Bonus FAQ (US).pdf'}, page_content='January 24, 2023 \n \n \n \n \n \n \n \n \n \n\uf0b7 \n\uf0b7 \n\uf0b7 \n\uf0b7'),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2023-01-24T18:23:58-08:00', 'author': 'KD984D', 'moddate': '2023-01-24T18:23:58-08:00', 'title': 'Microsoft Word - 2022 Success Bonus FAQ - US', 'source': 'documents\\2022 Success Bonus FAQ (US).pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'filename': '2022 Success Bonus FAQ (US).pdf'}, page_content='January 24, 2023'),
 Document(metadata={'source': 'documents\\DIRECTV Bereavement_Funeral Services FAQ 

In [ ]:
for document in final_documents:
    print(document.metadata["source"])
    print(document.page_content[:100])
    print("-" * 50)

documents\2022 Success Bonus FAQ (US).pdf
January 24, 2023 
 
 
 
 
 
 
 
 
 
 
 
 

--------------------------------------------------
documents\2022 Success Bonus FAQ (US).pdf
January 24, 2023
--------------------------------------------------
documents\DIRECTV Bereavement_Funeral Services FAQ Management_U.S._July 2023_FINAL.docx
Frequently Asked Questions – Bereavement and Funeral Paid Time Off
Management Employees, U.S. and Pu
--------------------------------------------------
documents\DIRECTV Bereavement_Funeral Services Policy Management_U.S._July 2023_FINAL.docx
Paid Time Off for Bereavement, Management Employees
U.S. and Puerto Rico 

Paid Time Off for Bereave
--------------------------------------------------
documents\DIRECTV Bereavement_Funeral Services Policy Management_U.S._July 2023_FINAL.docx
Employees in Canada

No
See Important Things to Know section of this policy.

Interns

Yes

Note:  T
--------------------------------------------------
documents\DIRECTV Bereav

In [7]:
## Vector Embedding And Vector StoreDB
from langchain_community.vectorstores import Chroma

db=Chroma.from_documents(final_documents,embeddings_1024)
db

In [84]:
### Retrieve the results from query vectorstore db
query="How do company-authorized holidays work if I am a part-time employee?"
retrieved_results=db.similarity_search(query, k=3)

In [85]:
print(retrieved_results)

[Document(metadata={'source': 'documents\\DIRECTV Holidays and Holiday Pay FAQ Management U.S._July 2023_FINAL.docx', 'filename': 'DIRECTV Holidays and Holiday Pay FAQ Management U.S._July 2023_FINAL.docx'}, page_content='Q: How do company-authorized holidays work if I am a part-time employee?\nA: You will receive holiday allowance for the number of hours you normally work. If you normally work a four-hour day, you receive four hours holiday allowance. If you normally work a 10-hour day, you receive ten hours holiday allowance.'), Document(metadata={'source': 'documents\\DIRECTV Holidays and Holiday Pay FAQ Management U.S._July 2023_FINAL.docx', 'filename': 'DIRECTV Holidays and Holiday Pay FAQ Management U.S._July 2023_FINAL.docx'}, page_content='*Occasionally the company may observe a holiday in a different manner, for example a Saturday holiday observed on a Monday, to accommodate the total number of holidays for a year and/or payroll needs.\nRefer to the Paid Time Off page of our E

In [83]:
for doc in retrieved_results:
    print(doc.metadata['source'])
    print(doc.page_content[:200])

documents\DIRECTV Paid Parental Leave Policy_ Management_Bargained U.S._07172023_FINAL.docx
Employees in San Francisco are required to exhaust all PPL before receiving any state or municipal paid parental leave benefits, if any. In states or municipalities where employees are entitled to and
documents\DIRECTV Paid Parental Leave Policy_ Management_Bargained U.S._07172023_FINAL.docx
Paid Parental Leave, Management and Bargained Employees
U.S. and Puerto Rico


Paid Parental Leave, Management and Bargained Employees
U.S. and Puerto Rico


Paid Parental Leave Policy

Adding a new m
documents\DIRECTV Paid Parental Leave Policy_ Management_Bargained U.S._07172023_FINAL.docx
You have 20 calendar days after submitting the leave request in Leavelink to submit the Paid Parental Leave Request Form along with the required proof of birth/adoption. If you do not provide the requ


In [8]:
retriever=db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

In [11]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Given a chat history and the latest user question,
            formulate a standalone question that can be
            understood without the chat history.

            Do NOT answer the question.
            Only reformulate it if needed.
            """
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

In [12]:
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

In [13]:
history_aware_retriever = (
    create_history_aware_retriever(
        llm,
        retriever,
        contextualize_q_prompt
    )
)

In [14]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an HR assistant.

            Answer only using the provided context.

            If the answer is not available,
            say so explicitly.

            Context:
            {context}
            """
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

In [16]:
question_answer_chain = (
    create_stuff_documents_chain(
        llm,
        qa_prompt
    )
)

In [17]:
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

In [20]:
response = rag_chain.invoke(
    {
        "input":
            "What are the application steps for it?",
        "chat_history":
            chat_history
    }
)

print(response["answer"])

According to the Employee Portal Paid Parental Leave page, the application steps for Paid Parental Leave are:

1. Submitting a Paid Parental Leave request form, with proof of birth or adoption to your supervisor (see Apply for PPL for acceptable documentation)
2. Supervisor approval and agreement to time off dates
3. Submitting the Paid Parental Leave time-off request in Leavelink (via Sedgewick), and supervisor approval notification from LeaveLink.


In [87]:
from langchain_groq import ChatGroq

groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

In [88]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

In [19]:
chat_history = [
    HumanMessage(
        content="What is PPL?"
    ),
    AIMessage(
        content="PPL is Paid Parental Leave"
    )
]

In [36]:
response=rag_chain.invoke("How do company-authorized holidays work if I am a part-time employee?")
docs = retriever.invoke("How do company-authorized holidays work if I am a part-time employee?")

sources = list({
    d.metadata['source']
    for d in docs
})

ValueError: The input to RunnablePassthrough.assign() must be a dict.

In [91]:
print(response.content)

print("\nSources:")
for source in sources:
    print("-", source)

You will receive holiday allowance for the number of hours you normally work. If you normally work a four-hour day, you receive four hours holiday allowance. If you normally work a 10-hour day, you receive ten hours holiday allowance.

Sources:
- documents\DIRECTV Holidays and Holiday Pay FAQ Management U.S._July 2023_FINAL.docx


In [92]:
from langchain_community.vectorstores import FAISS

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("documents/2022 Success Bonus FAQ (US).pdf")

docs = loader.load()

C:\Users\OmkarIngale\AppData\Local\Temp\ipykernel_63888\2606056547.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\OmkarIngale\Desktop\Training\RAG project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [102]:
docs = PyPDFLoader('documents/2022 Success Bonus FAQ (US).pdf').load()

In [6]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\OmkarIngale\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
)

In [9]:
from pdf2image import convert_from_path

pages = convert_from_path(
    "documents/2022 Success Bonus FAQ (US).pdf",
    dpi=400,
    poppler_path=r"C:\poppler\poppler-26.02.0\Library\bin"
)

for page in pages:
    text = pytesseract.image_to_string(page)
    print(text)

January 24, 2023

Frequently Asked Questions
2022 Success Bonus - US

. How will 2022 Success Bonus payments appear on pay slips?

Success Bonus appears as “Success Bonus” on US pay slips.

. What is “bonus eligible earnings?”

Bonus eligible earnings are the earnings on which the bonus award is based and includes things like
salary and hourly pay; overtime and double-time; shift & bilingual differentials and premium pay;
holiday, sick, vacation, PTO, jury duty and bereavement time; and some other earnings. It does not
include other bonuses, stipends, commissions, cash or non-cash awards, etc.

. Why are my earnings different than my current salary?
Earnings may be different than your current salary or annualized pay for a few reasons:

e For employees who worked the entire year, there was an additional paycheck in Workday with
our transition from semi-monthly to bi-weekly payroll, so in some cases, earnings might exceed
salary.

e Bonus eligible earnings include base wages but also ea

In [4]:
print(docs)

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2023-01-24T18:23:58-08:00', 'source': 'documents/2022 Success Bonus FAQ (US).pdf', 'file_path': 'documents/2022 Success Bonus FAQ (US).pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': 'Microsoft Word - 2022 Success Bonus FAQ - US', 'author': 'KD984D', 'subject': '', 'keywords': '', 'moddate': '2023-01-24T18:23:58-08:00', 'trapped': '', 'modDate': "D:20230124182358-08'00'", 'creationDate': "D:20230124182358-08'00'", 'page': 0}, page_content='January 24, 2023 \n \n \n \n \n \n \n \n \n \n\uf0b7 \n\uf0b7 \n\uf0b7 \n\uf0b7'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2023-01-24T18:23:58-08:00', 'source': 'documents/2022 Success Bonus FAQ (US).pdf', 'file_path': 'documents/2022 Success Bonus FAQ (US).pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': 'Microsoft Word - 2022 Success Bonus FAQ - US', 'author': 'KD984D', 'subject': '', 'keywords': ''

In [104]:
with open("debug_output.txt", "w", encoding="utf-8") as f:
    f.write(docs[0].page_content)

In [2]:
for i, doc in enumerate(docs):
    print("="*80)
    print(f"PAGE {i+1}")
    print(doc.page_content)

PAGE 1
January 24, 2023 
 
 
 
 
 
 
 
 
 
 
 
 

PAGE 2
January 24, 2023


In [10]:
import pymupdf4llm

text = pymupdf4llm.to_markdown(
    "documents/2022 Success Bonus FAQ (US).pdf"
)

print(text)

January 24, 2023 

**==> picture [95 x 36] intentionally omitted <==**

**==> picture [336 x 40] intentionally omitted <==**

**==> picture [458 x 79] intentionally omitted <==**

- 

- 

- 

- 

**==> picture [345 x 39] intentionally omitted <==**

January 24, 2023 

**==> picture [95 x 36] intentionally omitted <==**

**==> picture [337 x 149] intentionally omitted <==**

**==> picture [461 x 152] intentionally omitted <==**

**==> picture [384 x 38] intentionally omitted <==**




In [14]:
from langchain_core.documents import Document

def ocr_pdf(pdf_path):

    pages = convert_from_path(
        pdf_path,
        dpi=300,
        poppler_path=r"C:\poppler\poppler-26.02.0\Library\bin"
    )

    docs = []

    for page_num, page in enumerate(pages):

        text = pytesseract.image_to_string(
            page,
            config="--psm 6"
        )

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "page": page_num + 1
                }
            )
        )

    return docs

In [15]:
docs = ocr_pdf("documents/2022 Success Bonus FAQ (US).pdf")

In [10]:
from langchain_classic.chains import (
    create_history_aware_retriever,
    create_retrieval_chain
)

from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain
)

from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)